In [ ]:
# Convert to datetime objects
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])

# Extract time-based features
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek

# Calculate time_since_signup in seconds
fraud_df['time_since_signup'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds()

print("Time features created.")

In [ ]:
# Convert IP addresses to float for numerical range mapping
fraud_df['ip_address'] = fraud_df['ip_address'].astype(float)

# Sort both dataframes by the IP columns for merge_asof to work
fraud_df = fraud_df.sort_values('ip_address')
ip_df = ip_df.sort_values('lower_bound_ip_address')

# Merge using range-based lookup
merged_df = pd.merge_asof(fraud_df, ip_df, 
                          left_on='ip_address', 
                          right_on='lower_bound_ip_address', 
                          direction='backward')

# Handle IPs that fell outside the upper bound (they get mapped to 'Unknown')
merged_df.loc[merged_df['ip_address'] > merged_df['upper_bound_ip_address'], 'country'] = 'Unknown'
# Fill any remaining NaNs in country with 'Unknown'
merged_df['country'] = merged_df['country'].fillna('Unknown')

print("Geolocation integration complete.")

In [ ]:
# Drop columns not needed for modeling
X = merged_df.drop(['class', 'user_id', 'signup_time', 'purchase_time', 'device_id', 'ip_address', 'lower_bound_ip_address', 'upper_bound_ip_address'], axis=1, errors='ignore')
y = merged_df['class']

# One-hot encode categorical features (source, browser, sex, country)
X = pd.get_dummies(X, columns=['source', 'browser', 'sex', 'country'], drop_first=True)

# Normalize/scale numerical features
scaler = StandardScaler()
numerical_cols = ['purchase_value', 'age', 'time_since_signup']
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print("Encoding and Scaling complete.")